# Interventions — Exp4-style Plots on New Baseline Framework
Replicates the `exp4_id_plots` workflow using `exp3_section2_baselines` outputs (`id.jsonl`) grouped by baseline and depth.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve().parent
EXP3_ROOT = ROOT / 'results' / 'exp3_section2'

runs = sorted([p for p in EXP3_ROOT.glob('*') if p.is_dir()], reverse=True)
if not runs:
    raise RuntimeError(f'No runs found in {EXP3_ROOT}')

RUN_DIR = None
ID_PATH = None
for run in runs:
    candidate = run / 'id.jsonl'
    if candidate.exists() and candidate.stat().st_size > 0:
        RUN_DIR = run
        ID_PATH = candidate
        break

if RUN_DIR is None or ID_PATH is None:
    raise RuntimeError('No non-empty id.jsonl found in exp3_section2 runs')

print('Using run:', RUN_DIR)
print('ID file:', ID_PATH)

In [ ]:
rows = [json.loads(line) for line in ID_PATH.read_text().splitlines() if line.strip()]
df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('id.jsonl is empty')

df = df[df['id_estimate'].notna()].copy()
df['depth'] = df['depth'].astype(int)
print(df[['baseline', 'granularity', 'estimator']].drop_duplicates().sort_values(['baseline', 'granularity', 'estimator']).to_string(index=False))

In [ ]:
# Exp4-like depth curves for a selected granularity and estimator
GRANULARITY = 'last_token'
ESTIMATOR = 'twonn'

sel = df[(df['granularity'] == GRANULARITY) & (df['estimator'] == ESTIMATOR)].copy()
if sel.empty:
    raise RuntimeError('No rows for selected granularity/estimator')

agg = sel.groupby(['baseline', 'depth'], as_index=False)['id_estimate'].mean()

plt.figure(figsize=(8, 5))
for baseline, sub in agg.groupby('baseline'):
    sub = sub.sort_values('depth')
    plt.plot(sub['depth'], sub['id_estimate'], marker='o', label=baseline)

plt.title(f'ID vs depth ({GRANULARITY}, {ESTIMATOR})')
plt.xlabel('Depth')
plt.ylabel('Intrinsic dimension')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Baseline comparison at a fixed depth
DEPTH = int(sel['depth'].min())
bar = agg[agg['depth'] == DEPTH].sort_values('id_estimate')

plt.figure(figsize=(8, 4))
plt.barh(bar['baseline'], bar['id_estimate'])
plt.xlabel('Intrinsic dimension')
plt.title(f'Baseline comparison at depth={DEPTH} ({GRANULARITY}, {ESTIMATOR})')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()